# Malandra Pahlavi Dictionary Parser

In [18]:
# Import necessary libraries
import os
import re
import pandas as pd
from docx import Document

In [5]:
# Check if the file exists in the current folder
file_path = 'sample_malandra_dict.docx'
if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found in the current directory.")
else:
    # Read the Word document
    doc = docx.Document(file_path)
    print(f"Successfully loaded '{file_path}'")
    print(f"Document contains {len(doc.paragraphs)} paragraphs")
    
    # Print a sample of the first few paragraphs to verify content
    print("\nSample of document content:")
    for i, para in enumerate(doc.paragraphs[:3]):
        if para.text.strip():  # Skip empty paragraphs
            print(f"Paragraph {i+1}: {para.text[:70]}{'...' if len(para.text) > 70 else ''}")
            
            # Also show if it contains bold text (potential headword)
            has_bold = any(run.bold for run in para.runs)
            if has_bold:
                bold_text = ''.join(run.text for run in para.runs if run.bold)
                print(f"  Bold text: {bold_text}")

Successfully loaded 'sample_malandra_dict.docx'
Document contains 44 paragraphs

Sample of document content:
Paragraph 1: Narseh   /   / n. pr. of 1. a deity; 2. various rulers
  Bold text: Narseh
Paragraph 2: 	MMPers nrysh-yzd, Prth nrysf-yzd, Arm Nerseh; Av nairyō.saŋha-.
Paragraph 3: 	1.


In [26]:
# 1. Extracting Bold Headwords
# The first step is to identify entries by finding paragraphs that contain bold text. 
# The headword is typically the bold text at the beginning of a paragraph:

def extract_entries(docx_file):
    """
    Extract dictionary entries with bold headwords from a Word document,
    properly handling indented sub-entries with -, º, and ~ prefixes
    """
    # Load the document
    doc = Document(docx_file)
    
    entries = []
    current_main_headword = None  # Track the main entry headword
    current_headword = None
    current_text = []
    current_parent = None
    current_original_headword = None
    
    # Detect the chapter letter
    chapter = None
    
    # Scan the first few paragraphs for a chapter heading
    for i, para in enumerate(doc.paragraphs[:10]):  # Check only first 10 paragraphs
        text = para.text.strip()
        if not text:
            continue
        
        # Check if this might be a chapter heading (bold, single letter or short text)
        bold_segments = []
        for run in para.runs:
            if run.bold:
                bold_segments.append(run.text)
        
        bold_text = ''.join(bold_segments).strip()
        
        # If we found a bold letter or short text without slashes, it might be a chapter heading
        if bold_text and len(bold_text) <= 3 and '/' not in para.text:
            chapter = bold_text
            break
    
    # If no chapter found, use a default or derive from filename
    if not chapter:
        # Try to get chapter from filename (assuming format like "PHL-A.docx")
        filename = os.path.basename(docx_file)
        match = re.search(r'PHL-([A-Za-z])', filename)
        if match:
            chapter = match.group(1)
    
    # Now proceed with entry extraction
    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue
        
        # Check if paragraph contains bold text (potential headword)
        bold_segments = []
        has_bold = False
        
        for run in para.runs:
            if run.bold:
                has_bold = True
                bold_segments.append(run.text)
        
        if has_bold and bold_segments:
            # This paragraph contains a headword (either main entry or sub-entry)
            
            # Save the previous entry if there was one
            if current_headword:
                entries.append({
                    'headword': current_headword,
                    'parent': current_parent,
                    'original_headword': current_original_headword,
                    'entry_text': '\n'.join(current_text),
                    'chapter': chapter  # Add chapter information
                })
            
            # Extract the headword (all bold text concatenated)
            bold_text = ''.join(bold_segments).strip()
            
            # Skip if this is likely the chapter heading
            if bold_text == chapter and len(bold_text) <= 3 and '/' not in para.text:
                continue
                
            current_original_headword = bold_text  # Store the original headword
            
            # Check if this is a sub-entry (indented with -, º, or ~)
            is_sub_entry = False
            prefix_type = None
            
            # Look for indentation and prefix indicators in the paragraph
            if text.strip().startswith('-'):
                is_sub_entry = True
                prefix_type = 'hyphen'  # Prefix with hyphen (-)
            elif text.strip().startswith('º'):
                is_sub_entry = True
                prefix_type = 'join'    # Join directly (º)
            elif text.strip().startswith('~'):
                is_sub_entry = True
                prefix_type = 'space'   # Join with a space (~)
            
            # Handle the headword based on whether it's a main entry or sub-entry
            if is_sub_entry and current_main_headword:
                # This is a sub-entry, construct the full headword
                if prefix_type == 'hyphen':
                    current_headword = f"{current_main_headword}-{bold_text}"
                elif prefix_type == 'join':
                    current_headword = f"{current_main_headword}{bold_text}"
                elif prefix_type == 'space':
                    current_headword = f"{current_main_headword} {bold_text}"
                else:
                    current_headword = bold_text  # Fallback
                
                current_parent = current_main_headword  # Set parent relationship
            else:
                # This is a main entry
                current_headword = bold_text
                current_main_headword = bold_text  # Update the main entry
                current_parent = None  # Main entries have no parent
            
            # Get the rest of the text (non-bold parts) as the beginning of the entry
            non_bold_text = para.text
            for bold_segment in bold_segments:
                non_bold_text = non_bold_text.replace(bold_segment, '', 1)
            
            # Remove the prefix indicators from the text
            if is_sub_entry:
                if prefix_type == 'hyphen':
                    non_bold_text = re.sub(r'^\s*-\s*', '', non_bold_text)
                elif prefix_type == 'join':
                    non_bold_text = re.sub(r'^\s*º\s*', '', non_bold_text)
                elif prefix_type == 'space':
                    non_bold_text = re.sub(r'^\s*~\s*', '', non_bold_text)
            
            current_text = [non_bold_text.strip()]
        else:
            # This is a continuation of the current entry
            if current_headword:  # Only append if we have an active entry
                current_text.append(text)
    
    # Add the last entry
    if current_headword:
        entries.append({
            'headword': current_headword,
            'parent': current_parent,
            'original_headword': current_original_headword,
            'entry_text': '\n'.join(current_text),
            'chapter': chapter  # Add chapter information
        })
    
    return entries

In [27]:
# 2. Parsing Entry Components
# Once we have the entries, we parse each one to extract its components:

def parse_entry_components(entry):
    """Parse the components of a dictionary entry"""
    headword = entry['headword']
    parent = entry.get('parent')
    original_headword = entry.get('original_headword', headword)
    entry_text = entry['entry_text']
    chapter = entry.get('chapter')  # Get chapter information
    
    # Initialize components
    transliteration = ""
    pronunciation = ""
    definition = ""
    etymology = ""
    
    # Split by slashes to extract pronunciation
    parts = entry_text.split('/', 2)  # At most 2 splits (3 parts)
    
    if len(parts) >= 3:
        # First part contains transliteration (comes before the first slash)
        first_part = parts[0].strip()
        
        # Transliteration is typically at the end of the first part
        transliteration_match = re.search(r'(\S+)\s*$', first_part)
        if transliteration_match:
            transliteration = transliteration_match.group(1)
        
        # Second part is pronunciation
        pronunciation = parts[1].strip()
        
        # Third part contains definition and etymology
        remaining = parts[2].strip()
        
        # Etymology typically starts on a new line after the definition
        definition_parts = re.split(r'\n', remaining, 1)
        
        if len(definition_parts) > 1:
            definition = definition_parts[0].strip()
            etymology = definition_parts[1].strip()
        else:
            # If there's no clear separation by newline, try to identify etymology
            # by common markers like "MMPers", "NPers", "Av", etc.
            definition = remaining
            
            # Check for etymology markers
            etym_indicators = ["MMPers", "NPers", "Av", "OPers", "Prth", "Inscr", "Arab", "PhlGl"]
            for indicator in etym_indicators:
                if indicator in definition:
                    pattern = r'(.*?)(' + re.escape(indicator) + r'.*)'
                    match = re.search(pattern, definition, re.DOTALL)
                    if match:
                        definition = match.group(1).strip()
                        etymology = match.group(2).strip()
                        break
    else:
        # If there aren't enough slashes, just use the whole text as definition
        definition = entry_text
    
    return {
        'headword': headword,
        'parent': parent,
        'original_headword': original_headword,
        'is_sub_entry': parent is not None,
        'transliteration': transliteration,
        'pronunciation': pronunciation,
        'definition': definition,
        'etymology': etymology,
        'chapter': chapter  # Include chapter information
    }

In [28]:
# 3. Putting It All Together
# The main function brings everything together:

def process_dictionary(docx_file, output_csv=None):
    """Process a dictionary document and convert to structured format"""
    # Extract entries
    entries = extract_entries(docx_file)
    print(f"Found {len(entries)} entries")
    
    # Parse each entry
    parsed_entries = [parse_entry_components(entry) for entry in entries]
    
    # Convert to DataFrame
    result_df = pd.DataFrame(parsed_entries)
    
    # Reorder the columns to put parent information early
    column_order = [
        'headword', 
        'parent', 
        'original_headword', 
        'is_sub_entry', 
        'chapter',  # Add chapter to column order
        'transliteration', 
        'pronunciation', 
        'definition', 
        'etymology'
    ]
    
    # Make sure all columns exist before reordering
    existing_columns = [col for col in column_order if col in result_df.columns]
    result_df = result_df[existing_columns]
    
    # Save to CSV if output path provided
    if output_csv:
        result_df.to_csv(output_csv, index=False)
        print(f"Saved CSV file to {output_csv}")
    
    return result_df

In [29]:
# Now use these functions directly in your notebook
file_path = 'sample_malandra_dict.docx'
result_df = process_dictionary(file_path, 'pahlavi_dictionary.csv')

# Display the results
print("\nSample of parsed entries:")
display(result_df.head())  # This works in Jupyter to show a nice table view

# Specifically display sub-entries to verify they're being parsed correctly
sub_entries = result_df[result_df['is_sub_entry'] == True]
print(f"\nFound {len(sub_entries)} sub-entries")
if len(sub_entries) > 0:
    print("Sample of sub-entries:")
    display(sub_entries.head())
    
    # Show parent-child relationships for the first few entries
    print("\nSample parent-child relationships:")
    for i, row in sub_entries.head().iterrows():
        print(f"Entry: {row['headword']}")
        print(f"  Original: {row['original_headword']}")
        print(f"  Parent: {row['parent']}")
        print(f"  Definition: {row['definition'][:50]}...")

Found 14 entries
Saved CSV file to pahlavi_dictionary.csv

Sample of parsed entries:


,headword,parent,original_headword,is_sub_entry,chapter,transliteration,pronunciation,definition,etymology
0,Narseh,None,Narseh,False,None,,,n. pr. of 1. a deity; 2. various rulers,"MMPers nrysh-yzd, Prth nrysf-yzd, Arm Nerseh; ..."
1,nasA,None,nasA,False,None,aYFn,ns>y,"n. 'corpse, carrion'","MMPers n(y)s>h , NPers =; Av nasu-.\nPhlGl of ..."
2,nasA-brIn,nasA,brIn,True,None,,-PSKWN,n. 'corpse-rending',
3,nasA-keS,nasA,keS,True,None,,-kš,n. 'corpse-bearer',Av nasukaša-.\nPhlGl of nasukaša-; PRDD
4,nasA-kirrEn,nasA,kirrEn,True,None,,,n. 'corpse mutilator'\nPhlGl of nasu.kvrvt-; Dk7,



Found 6 sub-entries
Sample of sub-entries:


,headword,parent,original_headword,is_sub_entry,chapter,transliteration,pronunciation,definition,etymology
2,nasA-brIn,nasA,brIn,True,None,,-PSKWN,n. 'corpse-rending',
3,nasA-keS,nasA,keS,True,None,,-kš,n. 'corpse-bearer',Av nasukaša-.\nPhlGl of nasukaša-; PRDD
4,nasA-kirrEn,nasA,kirrEn,True,None,,,n. 'corpse mutilator'\nPhlGl of nasu.kvrvt-; Dk7,
5,nasA-nigAn,nasA,nigAn,True,None,,,adj. 'burying a corpse'\nPhlGl of nasuspā-.,
6,nasAIh,nasA,Ih,True,None,,,n. 'the burying of a corpse'\nPhlGl of nasuspa...,



Sample parent-child relationships:
Entry: nasA-brIn
  Original: brIn
  Parent: nasA
  Definition: n. 'corpse-rending'...
Entry: nasA-keS
  Original: keS
  Parent: nasA
  Definition: n. 'corpse-bearer'...
Entry: nasA-kirrEn
  Original: kirrEn
  Parent: nasA
  Definition: n. 'corpse mutilator'
PhlGl of nasu.kvrvt-; Dk7...
Entry: nasA-nigAn
  Original: nigAn
  Parent: nasA
  Definition: adj. 'burying a corpse'
PhlGl of nasuspā-....
Entry: nasAIh
  Original: Ih
  Parent: nasA
  Definition: n. 'the burying of a corpse'
PhlGl of nasuspaya-; ...


In [30]:
import os
import pandas as pd

# Define the input directory and output directory
input_dir = 'malandra'  # Directory containing the Word documents
output_dir = 'output'   # Directory to save the CSV files (will be created if it doesn't exist)

# Create the output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Get all Word files in the input directory (both .doc and .docx)
word_files = [f for f in os.listdir(input_dir) if f.endswith(('.doc', '.docx'))]

# Track all entries to create a master CSV
all_entries = []
successful_files = []
failed_files = []

# Process each document
for word_file in word_files:
    # Build full path to the input file
    input_path = os.path.join(input_dir, word_file)
    
    # Generate output CSV filename
    output_filename = os.path.splitext(word_file)[0] + '.csv'
    output_path = os.path.join(output_dir, output_filename)
    
    print(f"Processing {word_file}...")
    
    try:
        # Process the dictionary file
        result_df = process_dictionary(input_path, output_path)
        
        # Add source file information
        result_df['source_file'] = word_file
        
        # Append to the master list
        all_entries.append(result_df)
        
        # Track successful processing
        successful_files.append(word_file)
        
        # Display summary for this file
        print(f"  Found {len(result_df)} entries")
        sub_entries = result_df[result_df['is_sub_entry'] == True]
        print(f"  Sub-entries: {len(sub_entries)}")
        print(f"  Saved to {output_filename}\n")
    
    except Exception as e:
        # Track failed processing
        failed_files.append((word_file, str(e)))
        print(f"  Error processing {word_file}: {str(e)}\n")
        continue

# Combine all entries into a master CSV
if all_entries:
    master_df = pd.concat(all_entries, ignore_index=True)
    master_path = os.path.join(output_dir, 'pahlavi_dictionary_master.csv')
    master_df.to_csv(master_path, index=False)
    print(f"Created master dictionary with {len(master_df)} entries")
    print(f"Saved to {master_path}")

# Print summary of processing
print("\nProcessing summary:")
print(f"Successfully processed {len(successful_files)} files")
print(f"Failed to process {len(failed_files)} files")

if failed_files:
    print("\nFailed files:")
    for file_name, error in failed_files:
        print(f"  {file_name}: {error}")

print("\nProcessing complete!")

Processing PHL-Walf.docx...
Found 421 entries
Saved CSV file to output/PHL-Walf.csv
  Found 421 entries
  Sub-entries: 120
  Saved to PHL-Walf.csv

Processing PHL-A short (Bruce Lincoln's conflicted copy 2013-11-19).docx...
Found 746 entries
Saved CSV file to output/PHL-A short (Bruce Lincoln's conflicted copy 2013-11-19).csv
  Found 746 entries
  Sub-entries: 257
  Saved to PHL-A short (Bruce Lincoln's conflicted copy 2013-11-19).csv

Processing phl-kalf.docx...
Found 326 entries
Saved CSV file to output/phl-kalf.csv
  Found 326 entries
  Sub-entries: 48
  Saved to phl-kalf.csv

Processing PHL-Jalf.docx...
Found 81 entries
Saved CSV file to output/PHL-Jalf.csv
  Found 81 entries
  Sub-entries: 16
  Saved to PHL-Jalf.csv

Processing phl-b.docx...
Found 286 entries
Saved CSV file to output/phl-b.csv
  Found 286 entries
  Sub-entries: 81
  Saved to phl-b.csv

Processing PHL-Calf.docx...
Found 162 entries
Saved CSV file to output/PHL-Calf.csv
  Found 162 entries
  Sub-entries: 44
  Saved 

In [33]:
def fine_tune_dictionary(csv_file, output_file=None):
    """
    Fine-tune the dictionary CSV by adding part of speech and standardized transcription.
    
    Args:
        csv_file: Path to the input CSV file
        output_file: Path to save the enhanced CSV file (if None, will modify the input file)
    
    Returns:
        DataFrame with the enhanced dictionary
    """
    # Load the CSV file
    df = pd.read_csv(csv_file)
    print(f"Loaded {len(df)} entries from {csv_file}")
    
    # 1. Extract part of speech
    df['part_of_speech'] = df['definition'].apply(extract_part_of_speech)
    
    # 2. Generate standardized transcription
    df['transcription_skj'] = df['headword'].apply(standardize_transcription)
    
    # Reorder columns to include the new ones
    column_order = [
        'headword',
        'transcription_skj',
        'part_of_speech',
        'parent', 
        'original_headword', 
        'is_sub_entry', 
        'chapter',
        'transliteration', 
        'pronunciation', 
        'definition', 
        'etymology'
    ]
    
    # Make sure all columns exist before reordering
    existing_columns = [col for col in column_order if col in df.columns]
    df = df[existing_columns]
    
    # Save to CSV if output path provided
    if output_file:
        df.to_csv(output_file, index=False)
        print(f"Saved enhanced CSV file to {output_file}")
    else:
        # Overwrite the original file
        df.to_csv(csv_file, index=False)
        print(f"Updated the original CSV file {csv_file}")
    
    return df

def extract_part_of_speech(definition):
    """Extract part of speech from the definition"""
    if not isinstance(definition, str):
        return ""
    
    # Define part of speech patterns to search for
    pos_patterns = [
        (r'\bn\.', 'noun'),
        (r'\badj\.', 'adjective'),
        (r'\bvb\.', 'verb'),
        (r'\badv\.', 'adverb'),
        (r'\bprep\.', 'preposition'),
        (r'\bconj\.', 'conjunction'),
        (r'\bpron\.', 'pronoun'),
        (r'\binterj\.', 'interjection'),
        (r'\bnum\.', 'numeral'),
        (r'\bn\. pr\.', 'proper noun'),
        (r'\bp\. n\.', 'proper noun')
    ]
    
    # Search for each pattern
    for pattern, pos in pos_patterns:
        if re.search(pattern, definition):
            return pos
    
    return ""

def standardize_transcription(text):
    """Convert headword to standardized transcription with diacritical marks"""
    if not isinstance(text, str):
        return ""
    
    # Define replacement rules
    replacements = {
        'I': 'ī',
        'A': 'ā',
        'O': 'ō',
        'E': 'ē',
        'U': 'ū',  # Added the U > ū conversion
        'S': 'š',
        'š': 'š',  # Keep existing š unchanged
        'ī': 'ī',  # Keep existing ī unchanged
        'ā': 'ā',  # Keep existing ā unchanged
        'ō': 'ō',  # Keep existing ō unchanged
        'ē': 'ē',  # Keep existing ē unchanged
        'ū': 'ū'   # Keep existing ū unchanged
    }
    
    # Apply replacements
    result = ''.join(replacements.get(c, c) for c in text)
    
    return result



In [39]:
def extract_etymologies(text):
    """Extract language-specific etymologies into separate columns"""
    if not isinstance(text, str) or not text:
        return {}
    
    # Define the language mappings (abbreviation to column name)
    language_map = {
        'NPers': 'New_Persian',
        'JPers': 'Judeo_Persian',
        'Prth': 'Parthian',
        'MPrth': 'Manich_Parth',
        'MMPers': 'Manich_MP',
        'Av': 'Avestan',
        'OPers': 'Old_Persian',
        'Aram': 'Aramaic',
        'Prt': 'Prakrit',
        'Arab': 'Arabic',
        'Bal': 'Baluchi',
        'Syr': 'Syriac',
        'OInd': 'Old_Indian',
        'Inscr': 'Inscription',
        'PhlGl': 'Pahlavi_Gloss',
        'Skt': 'Sanskrit'  # Added Sanskrit
    }
    
    # Initialize empty results dictionary
    results = {col: '' for col in language_map.values()}
    
    # Find language abbreviations and their associated content
    for abbr, col_name in language_map.items():
        # Look for the abbreviation followed by content up to the next language abbreviation or end
        pattern = r'(?:^|[\s;.,])({})\s+(.*?)(?=\s+(?:{})|$|[;.])'.format(
            re.escape(abbr), 
            '|'.join(re.escape(a) for a in language_map.keys())
        )
        
        matches = re.finditer(pattern, text, re.DOTALL)
        
        # Collect all matches for this language (there might be multiple)
        contents = []
        for match in matches:
            content = match.group(2).strip()
            # Remove trailing punctuation
            content = re.sub(r'[;.,]\s*$', '', content)
            if content:
                contents.append(content)
        
        # Join all contents for this language
        if contents:
            results[col_name] = '; '.join(contents)
    
    return results

# Apply this function to your DataFrame
def add_etymology_columns(df):
    """Add columns for specific etymologies, checking both etymology and definition columns"""
    # Get all unique language columns from the entire dataset
    all_language_columns = set()
    etymology_data = []
    
    print("Extracting etymological information...")
    
    # Process each row
    for idx, row in df.iterrows():
        # Combine etymology and definition to look for language information
        etymology = row.get('etymology', '')
        definition = row.get('definition', '')
        combined_text = f"{etymology} {definition}"
        
        # Extract etymology information
        etym_dict = extract_etymologies(combined_text)
        all_language_columns.update(etym_dict.keys())
        etymology_data.append(etym_dict)
    
    print(f"Found {len(all_language_columns)} language columns")
    
    # Add each language column to the DataFrame
    for column in sorted(all_language_columns):
        df[column] = [data.get(column, '') for data in etymology_data]
    
    return df

In [40]:
# Path to your master CSV
master_csv = "output/pahlavi_dictionary_master.csv"

# Path for the enhanced output
enhanced_csv = "output/pahlavi_dictionary_enhanced.csv"

# Load the CSV file
df = pd.read_csv(master_csv)
print(f"Loaded {len(df)} entries from {master_csv}")

# 1. Extract part of speech
df['part_of_speech'] = df['definition'].apply(extract_part_of_speech)

# 2. Generate standardized transcription
df['transcription_skj'] = df['headword'].apply(standardize_transcription)

# 3. Extract etymological information into separate columns
df = add_etymology_columns(df)

# Move the language columns to the end of the DataFrame
# Get all column names
all_columns = list(df.columns)

# Identify which ones are etymology columns
language_columns = [col for col in all_columns if col in [
    'New_Persian', 'Judeo_Persian', 'Parthian', 'Manich_Parth', 
    'Manich_MP', 'Avestan', 'Old_Persian', 'Aramaic', 'Prakrit',
    'Arabic', 'Baluchi', 'Syriac', 'Old_Indian', 'Inscription',
    'Pahlavi_Gloss'
]]

# Create a list of columns with etymology columns at the end
core_columns = [col for col in all_columns if col not in language_columns]
reordered_columns = core_columns + language_columns

# Reorder the DataFrame
df = df[reordered_columns]

# Save to CSV
df.to_csv(enhanced_csv, index=False)
print(f"Saved enhanced CSV file to {enhanced_csv}")

# Display sample results with etymology columns
print("\nSample of enhanced entries with etymology information:")
display(df.head())

# Count entries with each etymology language
etymology_stats = {}
for col in language_columns:
    etymology_stats[col] = sum(df[col] != '')

print("\nEtymology statistics (entries per language):")
for lang, count in sorted(etymology_stats.items(), key=lambda x: x[1], reverse=True):
    print(f"{lang}: {count}")

Loaded 7839 entries from output/pahlavi_dictionary_master.csv
Extracting etymological information...
Found 16 language columns
Saved enhanced CSV file to output/pahlavi_dictionary_enhanced.csv

Sample of enhanced entries with etymology information:


,headword,parent,original_headword,is_sub_entry,chapter,transliteration,pronunciation,definition,etymology,source_file,...,Judeo_Persian,Manich_MP,Manich_Parth,New_Persian,Old_Indian,Old_Persian,Pahlavi_Gloss,Parthian,Prakrit,Syriac
0,w n,NaN,w n,False,W,NaN,NaN,NaN,NaN,PHL-Walf.docx,...,,,,,,,,,,
1,wabz,NaN,wabz,False,W,"cpn,_zpn","wpc, wpz",n. zool. 'wasp',Bal gwabz; Av waBZaka-.,PHL-Walf.docx,...,,,,,,,,,,
2,wacast,NaN,wacast,False,W,nTFcn,wcst',n. 'strophe (of the Gathas)',Av wacastaSti-.\nPhlGl of wacastaSti-,PHL-Walf.docx,...,,,,,,,of wacastaSti- n,,,
3,waccag,NaN,waccag,False,W,"nkcn,_nkcb","wck', bck'","n. 'child, baby, cub'",NPers baccah; cf. Prt awacca- < Skt apatya-. T...,PHL-Walf.docx,...,,,,baccah,,,,,awacca- <,
4,wad,NaN,wad,False,W,"DELII_,_ntn","SLY>, wt'","adj. 'bad, evil'",NaN,PHL-Walf.docx,...,,,,,,,,,,



Etymology statistics (entries per language):
New_Persian: 1949
Avestan: 1370
Manich_MP: 1109
Pahlavi_Gloss: 498
Aramaic: 301
Old_Persian: 300
Manich_Parth: 295
Old_Indian: 178
Parthian: 178
Arabic: 51
Judeo_Persian: 46
Inscription: 32
Syriac: 24
Baluchi: 18
Prakrit: 1


## Parse Skjaervo textbook examples

In [106]:
import re
import csv

def direct_asterisk_quote_parser(input_file, output_csv):
    """
    A direct parser that follows this exact logic:
    1. When you see an asterisk, continue until you get to an asterisk followed by double quote
    2. Save everything between asterisks as the original
    3. Save everything in quotes as the translation
    4. Include any parenthetical text after the quotes in the translation
    
    Args:
        input_file: Path to the input file
        output_csv: Path to save the extracted translation pairs
    """
    print(f"Reading file: {input_file}")
    
    # Read the file as one long string
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            text = f.read()
    except UnicodeDecodeError:
        try:
            with open(input_file, 'r', encoding='latin-1') as f:
                text = f.read()
        except Exception as e:
            print(f"Error reading file: {e}")
            return []
    
    # Find all *original* "translation" patterns with possible following parentheses
    # This regex captures:
    # 1. Text between asterisks (*original*)
    # 2. Text between quotes ("translation")
    # 3. Optional text in parentheses that follows the quotes
    pattern = r'\*(.*?)\*\s*"(.*?)"(\s*\([^)]*\))?'
    
    pairs = []
    
    # Find all matches in the text
    for match in re.finditer(pattern, text):
        original = match.group(1).strip()
        translation = match.group(2).strip()
        
        # If there's parenthetical text, add it to the translation
        if match.group(3):
            translation += " " + match.group(3).strip()
        
        # Skip if original contains "Note" or "Dictionary" or is very short
        if ("Note" in original or 
            "Dictionary" in original or 
            "MacKenzie" in original or
            len(original) < 2):
            continue
            
        pairs.append((original, translation))
    
    print(f"Found {len(pairs)} translation pairs")
    
    # Remove duplicates
    unique_pairs = []
    seen = set()
    
    for original, translation in pairs:
        # Create a key for deduplication
        key = f"{original}|||{translation}"
        
        if key not in seen:
            seen.add(key)
            unique_pairs.append((original, translation))
    
    print(f"After removing duplicates: {len(unique_pairs)} unique pairs")
    
    # Write to CSV
    with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile, quoting=csv.QUOTE_ALL)
        writer.writerow(["Original", "Translation"])
        writer.writerows(unique_pairs)
    
    print(f"Wrote {len(unique_pairs)} pairs to {output_csv}")
    
    # Print sample pairs
    print("\nSample extracted pairs:")
    for i, (original, translation) in enumerate(unique_pairs[:10]):
        print(f"{i+1}. Original: {original}")
        print(f"   Translation: {translation}")
    
    return unique_pairs

# Input and output files
input_file = "cleaned_example_sections.md"  # Use the cleaned file
output_csv = "direct_parsed_pairs.csv"

# Run the function
direct_asterisk_quote_parser(input_file, output_csv)

Reading file: cleaned_example_sections.md
Found 185 translation pairs
After removing duplicates: 168 unique pairs
Wrote 168 pairs to direct_parsed_pairs.csv

Sample extracted pairs:
1. Original: ay mard tō kē hē
   Translation: O man, who are you?
2. Original: an Kay Husrōy* *ham* = *an ham Kay Husrōy
   Translation: I am Kay Husrōy
3. Original: ašmāh kē hēd
   Translation: Who are you (all)?
4. Original: amāh mazdēsn hēm
   Translation: We are Mazdayasnians
5. Original: ašmāh driyōš mardōm hēd
   Translation: you (all) are poor people
6. Original: awēšān mard hēnd zan nē hēnd
   Translation: they are men, not women
7. Original: dēw hēnd nē mardōm
   Translation: they are demons, not humans.
8. Original: pad gētīy nē būd ham
   Translation: I have not (always) been in this world
9. Original: andar mān mār būd
   Translation: (there) was a snake in the house
10. Original: dušdānāg dēw-ēd
   Translation: you are stupid demons


[('ay mard tō kē hē', 'O man, who are you?'),
 ('an Kay Husrōy* *ham* = *an ham Kay Husrōy', 'I am Kay Husrōy'),
 ('ašmāh kē hēd', 'Who are you (all)?'),
 ('amāh mazdēsn hēm', 'We are Mazdayasnians'),
 ('ašmāh driyōš mardōm hēd', 'you (all) are poor people'),
 ('awēšān mard hēnd zan nē hēnd', 'they are men, not women'),
 ('dēw hēnd nē mardōm', 'they are demons, not humans.'),
 ('pad gētīy nē būd ham', 'I have not (always) been in this world'),
 ('andar mān mār būd', '(there) was a snake in the house'),
 ('dušdānāg dēw-ēd', 'you are stupid demons'),
 ('-ēw', 'one'),
 ('mardōm', 'people, humans'),
 ('ēn mardōm kadām hēnd', 'which (= who) are these people?'),
 ('ēn kanīg ō abārīgān kanīgān nē homānāg',
  'this young woman is not like (other) young women'),
 ('Zarduxšt pad mardōmān rād ud rāst būd',
  'Zarduxšt was generous and straight/truthful toward people'),
 ('was', 'much, many, very'),
 ('andar ērān šahr was mazdēsn būd hēnd',
  'in the land of the Iranians (= Iran), there were many 